## SAT-style Question Answering with GGUF Models

Learning objective: Load a small model from a shared directory and use it to answer SAT-style questions with a small language model.

## What this notebook teaches
- how to locate a shared folder of GGUF weights on your laptop or hub
- how to load a model with llama-cpp-python and walk through a tutor-style prompt
- how to inspect the model's answer and reasoning with reflection checkpoints
- how to compare the model's reasoning across random, filtered, and batched SAT items

## Where the questions come from
- We download the PineSAT Questionbank API  at https://pinesat.com/api/questions.
- PineSAT hosts community-built SAT-style questions so you can practice with authentic formats without licensing hurdles.
- The endpoint replies in JSON (JavaScript Object Notation, a plain text key-value format) so we can inspect passages, choices, and answer keys with simple loops.

## How to navigate the lesson
1. Every markdown cell previews the next code cell so you always know why a command matters.
2. Reflection checkpoints append your thoughts to answers.txt so you can track how your understanding changes.
3. Later cells batch four easy questions into a table so you can see accuracy trends without extra plotting.

Tip: Keep an eye on the model context size (the amount of text it can read at once) and thread count so you do not overload a shared machine.

In [ ]:

from llama_cpp import Llama  # loads llama-cpp-python so we can run GGUF models
import os  # lets us work with file paths
import random  # lets us pick random items from a list
import requests  # lets us call web APIs over HTTP
import pandas as pd  # helps with working with tables and spreadsheets
import re  # lets us search text with patterns
from IPython.display import display  # lets us show interactive widgets inside the notebook
import ipywidgets as widgets  # adds dropdown menus and buttons for simple UI


Check which shared directory is available on this machine. If you are on a hub, the shared directory is usually under /home/jovyan. On a laptop it might live in your Documents folder.

In [ ]:

possible_directories = [
    "/home/jovyan/shared/",
    "/home/jovyan/shared_readwrite/",
    "/Users/ericvandusen/SmallLM/Models"
]

existing_directories = []
for directory_path in possible_directories:
    if os.path.exists(directory_path):
        print("Found possible directory:", directory_path)
        existing_directories.append(directory_path)
    else:
        print("Did not find:", directory_path)


Pick a directory to use. We default to the first path that exists, and you can type a different one if you want.

In [ ]:

if len(existing_directories) > 0:
    model_directory = existing_directories[0]
    print("Using this directory by default:", model_directory)
else:
    model_directory = input("Type a directory path that contains your .gguf files: ")

print("Current model directory:", model_directory)


List every .gguf model file in the chosen directory so we can pick any model that is available. Use the dropdown to make your choice before running the loader cell.

In [ ]:

available_models = []
for filename in os.listdir(model_directory):
    if filename.endswith(".gguf"):
        available_models.append(filename)

if len(available_models) == 0:
    print("No .gguf files found in", model_directory)
else:
    print("Models found in", model_directory)
    dropdown_default = available_models[0]
    for candidate_name in available_models:
        lowercase_name = candidate_name.lower()
        if "qwen" in lowercase_name:
            dropdown_default = candidate_name
            break
    model_dropdown = widgets.Dropdown(
        options=available_models,
        description="Model:",
        value=dropdown_default
    )
    display(model_dropdown)
    print("Use the dropdown to pick a model, then run the next cell to load it.")


Checkpoint #1: Look up the model card for the model you picked - what was that mode trained on? What is its context size? How many threads does it use by default?

Load the selected model with llama-cpp-python so it is ready to answer questions.

In [ ]:
selected_model_name = model_dropdown.value
model_path = os.path.join(model_directory, selected_model_name)


In [ ]:

model = Llama(
    model_path=model_path,
    n_ctx=2048,
    n_threads=4,
    verbose=False
)



Warm-up: ask a simple SAT-style algebra question to confirm that the model responds.

Here is a simple question to get us started. We will ask the model to reason step by step and put its final answer in a box. This is a common format for SAT questions, and it encourages the model to show its work.  We will have 
-  an algebraic equation to solve, and
-  four answer choices to pick from.
- a tutor-style prompt that encourages the model to explain its reasoning clearly.


In [ ]:
# Build the Question from parts 

warmup_question = "If 3x + 5 = 14, what is the value of x?"
warmup_choices = "A) 2\nB) 3\nC) 4\nD) 5"
warmup_prompt = """
Here is an SAT-style multiple-choice question:
Question: {question_text}
Choices: {choices_text}

Reason step by step. On the very last line of your response, write exactly:
Final answer: X
where X is the letter of your choice (A, B, C, or D).
"""

# Put the parts together into the format we want to send to the model
messages = []
messages.append({"role": "system", "content": "You are a helpful math tutor who explains each step clearly."})
messages.append({"role": "user", "content": warmup_prompt.format(question_text=warmup_question, choices_text=warmup_choices)})

# Send the question to the model and print the response
warmup_response = model.create_chat_completion(messages=messages)
# this next line pulls the text of the model's response out of the full response object that the model returns, so we can print just the text

warmup_text = warmup_response["choices"][0]["message"]["content"]
print(warmup_text)

## Source an open source set of SAT-style questions
Download the SAT Questionbank from PineSAT so we can pull authentic practice questions.] We will use the API endpoint at https://pinesat.com/api/questions, which returns a JSON object with passages, questions, choices, and answer keys. We can loop through this data to inspect the format and content of the questions.

In [ ]:
base_url = "https://pinesat.com/api/questions"

english_questions = requests.get(base_url, params={"section": "english"}).json()
math_questions = requests.get(base_url, params={"section": "math"}).json()

english_nested = pd.DataFrame(english_questions)
math_nested = pd.DataFrame(math_questions)

print("English questions:", len(english_nested))
print("Math questions:", len(math_nested))

In [ ]:
english_questions[0]

In [ ]:
english_questions[0]["question"]

In [ ]:
english_questions[0]["question"]['question']

In [ ]:
english_questions[0]["question"]['choices']

In [ ]:
english_questions[0]["question"]['paragraph']

In [ ]:
english_questions[0]["question"]['correct_answer']

In [ ]:
english_details = pd.json_normalize(english_nested["question"])
english_details.head()

In [ ]:
english_df = pd.concat(
    [english_df.drop(columns=["question"]).reset_index(drop=True),
     english_details.reset_index(drop=True)],
    axis=1
)

english_df.head()

In [ ]:
# adapt the code for math 

math_details = pd.json_normalize(math_nested["question"])
math_details.head()
math_df = pd.concat(
    [math_df.drop(columns=["question"]).reset_index(drop=True),
     math_details.reset_index(drop=True)],
    axis=1
)
math_df.head()

In [ ]:
english_df

Pick a random question from the bank without filtering so you can see the full structure.

In [ ]:
# Pick a math question from the question bank that we can test on
math_questions[0]



In [ ]:
math_df.head()

### Ask the model - English
We will now pass the random question to the model with the same tutor-style prompt we used for the warm-up question. This allows us to compare how the model handles a random question versus a simple one.

Ask the model to answer the random question. The same friendly tutor prompt is reused so you can compare responses.

In [ ]:

# Pick a random question from the English question bank
random_index = random.randint(0, len(english_questions) - 1)
random_entry = english_questions[random_index]

# Pull out the question text and choices
random_question_text = random_entry["question"]['question']
random_choices_text = random_entry["question"]['choices']

print("Question:", random_question_text)
print("Choices:", random_choices_text)

# Ask the model to answer it
random_messages = []
random_messages.append({"role": "system", "content": "You are a helpful tutor who explains each step clearly."})
random_messages.append({"role": "user", "content": warmup_prompt.format(question_text=random_question_text, choices_text=random_choices_text)})

random_response = model.create_chat_completion(messages=random_messages)
random_answer_text = random_response["choices"][0]["message"]["content"]
print(random_answer_text)


## Ask the model - Math
Now let's repeat the process with a random math question from the bank. This allows us to compare how the model handles different subjects and question formats.

In [ ]:
# Pick a random question from the Math  question bank
random_index = random.randint(0, len(math_questions) - 1)
random_entry = math_questions[random_index]

# Pull out the question text and choices
random_question_text = random_entry["question"]['question']
random_choices_text = random_entry["question"]['choices']

print("Question:", random_question_text)
print("Choices:", random_choices_text)

# Ask the model to answer it
random_messages = []
random_messages.append({"role": "system", "content": "You are a helpful tutor who explains each step clearly."})
random_messages.append({"role": "user", "content": warmup_prompt.format(question_text=random_question_text, choices_text=random_choices_text)})

random_response = model.create_chat_completion(messages=random_messages)
random_answer_text = random_response["choices"][0]["message"]["content"]
print(random_answer_text)

Checkpoint #2: Why are we testing the model with the domains provided in the bank? Write a short explanation.

### Build a mini SAT practice set


Lets build a set of test question that we can then pass to the model in a batch. This allows us to see how the model performs across multiple questions and identify any patterns in its strengths or weaknesses.

Filter the bank by difficulty and subject so you can target specific skills.

In [ ]:

difficulty_widget = widgets.Dropdown(
    options=["Easy", "Medium", "Hard"],
    value="Easy",
    description="Difficulty:"
)

section_widget = widgets.Dropdown(
    options=["English", "Math"],
    value="English",
    description="Section:"
)

num_questions_widget = widgets.Dropdown(
    options=[ 2, 4, 8, 10, 12],
    value=4,
    description="# Questions:"
)

display(difficulty_widget)
display(section_widget)
display(num_questions_widget)
print("Choose your settings above, then run the next cell to test the model.")


Now send each question to the model and collect the results in a table.

In [ ]:
# Read the widget values
chosen_difficulty = difficulty_widget.value
chosen_section = section_widget.value
chosen_count = num_questions_widget.value

# Pick the right question list based on section
if chosen_section == "English":
    question_pool = english_questions
else:
    question_pool = math_questions

# Filter by difficulty and shuffle so we get a fresh sample each run
filtered_questions = []
for question_entry in question_pool:
    if question_entry.get("difficulty", "").lower() == chosen_difficulty.lower():
        filtered_questions.append(question_entry)

random.shuffle(filtered_questions)
practice_set = filtered_questions[:chosen_count]

# Pull out question text, choices, paragraph, and correct answer for each item
practice_questions = []
practice_choices = []
practice_paragraphs = []
practice_answers = []
for question_entry in practice_set:
    practice_questions.append(question_entry["question"]["question"])
    practice_choices.append(question_entry["question"]["choices"])
    practice_paragraphs.append(question_entry["question"].get("paragraph", ""))
    practice_answers.append(question_entry["question"].get("correct_answer", ""))

print("Built a practice set of", len(practice_questions), chosen_difficulty, chosen_section, "questions.")
for item_index in range(len(practice_questions)):
    print("\nQ" + str(item_index + 1) + ":", practice_questions[item_index])
    if len(practice_paragraphs[item_index]) > 0:
        print("Passage:", practice_paragraphs[item_index])
    print("Choices:", practice_choices[item_index])
    print("Correct answer:", practice_answers[item_index])


Now send each question to the model and collect the results in a table.

In [ ]:
batch_results = []
for item_index in range(len(practice_questions)):
    # If there is a passage, include it before the question
    passage_text = practice_paragraphs[item_index]
    if len(passage_text) > 0:
        full_question_text = "Passage: " + passage_text + "\n\n" + practice_questions[item_index]
    else:
        full_question_text = practice_questions[item_index]

    batch_messages = []
    batch_messages.append({"role": "system", "content": "You are a helpful tutor who explains each step clearly. Always end your response with exactly: Final answer: X where X is the letter A, B, C, or D."})
    batch_messages.append({"role": "user", "content": warmup_prompt.format(question_text=full_question_text, choices_text=practice_choices[item_index])})

    batch_response = model.create_chat_completion(messages=batch_messages)
    batch_reply_text = batch_response["choices"][0]["message"]["content"]

    extracted_answer = ""
    answer_match = re.search(r"[Ff]inal answer:\s*([A-D])", batch_reply_text)
    if answer_match is not None:
        extracted_answer = answer_match.group(1)

    is_correct = False
    if len(practice_answers[item_index]) > 0 and len(extracted_answer) > 0:
        is_correct = extracted_answer.strip().upper().startswith(practice_answers[item_index].upper())

    batch_results.append({
        "Section": chosen_section,
        "Difficulty": chosen_difficulty,
        "Question": practice_questions[item_index],
        "Correct Answer": practice_answers[item_index],
        "Model Guess": extracted_answer,
        "Correct?": is_correct
    })

batch_results_table = pd.DataFrame(batch_results)
batch_results_table

Checkpoint #3: Try an easy question from a subject you choose. Did the model get it right? Explain how you checked.

In [ ]:

student_reply_three = input("Describe what you tried and whether the model's answer matched the key.
")
with open('answers.txt', 'a') as answer_file:
    answer_file.write(student_reply_three)
    answer_file.write('
')


Print a summary showing how many questions the model got right overall.

In [ ]:

correct_count = 0
for result_row in batch_results:
    if result_row["Correct?"] == True:
        correct_count = correct_count + 1

print("Score:", correct_count, "out of", len(batch_results))
print(batch_results_table)


Summary: You located a shared model directory, picked any GGUF file, loaded it with llama-cpp-python, and exercised it on SAT-style questions with and without filters. You also logged your reflections in answers.txt.